In [ ]:
import pandas as pd



trd_errors_data = pd.read_csv(r'.\tird_dns_pipeline_errors.csv')
trd_ns_data = pd.read_csv(r'.\tird_dns_pipeline_multi_ns.csv')

second_errors_data = pd.read_csv(r'.\second_dns_pipeline_errors.csv')
second_ns_data = pd.read_csv(r'.\second_dns_pipeline_multi_ns.csv')

main_ns_data = pd.read_csv(r'.\main_dns_pipeline_multi_ns.csv')

trd_cleaned = trd_ns_data[~trd_ns_data['domain'].isin(trd_errors_data['domain'])]

second_cleaned = second_ns_data[~second_ns_data['domain'].isin(second_errors_data['domain'])]
final_cleaned = main_ns_data[~main_ns_data['domain'].isin(second_ns_data['domain'])]


#get company_names
main_company_names_full = pd.read_csv(r'.\main_companies_ranked_by_revenue.csv')
trd_company_names_full = pd.read_csv(r'.\tird_domain_companyname.csv')

#add company_name column from 
def add_company_names(df, company_names_df):
    df = df.merge(company_names_df[['domain', 'company_name']], on='domain', how='left')
    return df


#merge all cleaned data
all_cleaned = pd.concat([trd_cleaned, second_cleaned, final_cleaned])

all_cleaned = add_company_names(all_cleaned, trd_company_names_full)
all_cleaned = add_company_names(all_cleaned, main_company_names_full)



#existem 2 colunas company_name_x e company_name_y, preciso unificar as 2 mantendo os valores nao nulos e se forem iguais, deixar um unico valor
all_cleaned['company_name'] = all_cleaned['company_name_x'].combine_first(all_cleaned['company_name_y'])

all_cleaned = all_cleaned.drop(columns=['company_name_x', 'company_name_y'])

all_cleaned.to_csv(r'.\all_dns_pipeline_multi_ns_cleaned.csv', index=False)

In [ ]:
#Listar company names que estão no NOMS-2026\src\main_companies_ranked_by_revenue.csv e que não estão no all_cleaned

main_company_names_full = pd.read_csv(r'.\main_companies_ranked_by_revenue.csv')
missing_companies = main_company_names_full[~main_company_names_full['company_name'].isin(all_cleaned['company_name'])]

missing_companies.to_csv(r'.\missing_companies_dns_pipeline.csv', index=False)

In [ ]:
import pandas as pd

#incluir colunas de revenue, country_name e sector e rank da planilha main_companies_ranked_by_revenue.csv no all_dns_pipeline_multi_ns_cleaned.csv

all_cleaned = pd.read_csv(r'.\all_dns_pipeline_multi_ns_cleaned.csv')
main_company_names_full = pd.read_csv(r'.\main_companies_ranked_by_revenue.csv')

all_cleaned = all_cleaned.merge(main_company_names_full[['company_name', 'revenue', 'country_name', 'sector', 'rank']], on='company_name', how='left')

#sort por revenue decrescente
all_cleaned = all_cleaned.sort_values(by='revenue', ascending=False)

#tirar ".0" no final do campo revenue e transformar em int
all_cleaned['revenue'] = all_cleaned['revenue'].fillna(0).astype(int)


all_cleaned.to_csv(r'.\all_dns_pipeline_multi_ns_cleaned_with_revenue.csv', index=False)

In [ ]:
import pandas as pd

sorting_df = pd.read_csv(r'.\all_dns_pipeline_multi_ns_cleaned_with_revenue.csv')

#sort by multiple columns: revenue (desc)
sorting_df = sorting_df.sort_values(by=['revenue'], ascending=[False])    
sorting_df.to_csv(r'.\all_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv', index=False)

In [ ]:
import pandas as pd

#list unique companies in all_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv WHERE REVENUE IS > 0
all_cleaned = pd.read_csv(r'.\all_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv')
unique_companies = all_cleaned[all_cleaned['revenue'] > 0]['company_name'].dropna().unique()
unique_companies_df = pd.DataFrame(unique_companies, columns=['company_name'])
unique_companies_df.to_csv(r'.\unique_companies_dns_pipeline.csv', index=False)

In [ ]:
import pandas as pd

#listar as empresas que possuem revenue >0 em companies_ranked_by_revenue.csv e que não estão em unique_companies_dns_pipeline.csv
companies_ranked = pd.read_csv('./companies_ranked_by_revenue.csv')
unique_companies = pd.read_csv('./unique_companies_dns_pipeline.csv')

# Fill NaN values in the revenue column with 0 before converting to integers
companies_ranked['revenue'] = companies_ranked['revenue'].fillna(0).astype(int)

filtered_companies = companies_ranked[(companies_ranked['revenue'] > 0) & (~companies_ranked['company_name'].isin(unique_companies['company_name']))]
filtered_companies.to_csv(r'./missing_companies_from_dns_pipeline.csv', index=False)

In [ ]:
import pandas as pd

#importar dns_pipeline_multi_ns.csv e retirar as linhas que possuem domain presente em dns_pipeline_errors.csv

ns_data = pd.read_csv(r'.\dns_pipeline_multi_ns.csv')
errors_data = pd.read_csv(r'.\dns_pipeline_errors.csv')

cleaned_data = ns_data[~ns_data['domain'].isin(errors_data['domain'])]

display(cleaned_data)

#unir cleaned_data com companies_ranked_by_revenue.csv para adicionar revenue,country_name,sector,rank usando domain
companies_ranked = pd.read_csv(r'.\companies_ranked_by_revenue.csv')
cleaned_data = cleaned_data.merge(companies_ranked[['domain', 'company_name', 'revenue', 'country_name', 'sector', 'rank']], on='domain', how='left')

display(cleaned_data)

cleaned_data.to_csv(r'.\tttdns_pipeline_multi_ns_cleaned_with_revenue.csv', index=False)

#dar append no arquivo all_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv

all_cleaned = pd.read_csv(r'.\all_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv')
final_data = pd.concat([all_cleaned, cleaned_data])
final_data = final_data.sort_values(by='revenue', ascending=False)
final_data.to_csv(r'.\final_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv', index=False)

In [36]:
import pandas as pd

#list unique company_names, domains, revenue, country_name, sector and rank in final_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv WHERE REVENUE IS > 0
final_data = pd.read_csv(r'.\final_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv')
unique_companies = final_data[final_data['revenue'] > 0][['company_name', 'domain', 'revenue', 'country_name', 'sector', 'rank']].drop_duplicates()

# drop duplicates based on company_name, keeping the first occurrence (which has the highest revenue due to sorting)
unique_companies = unique_companies.sort_values('revenue', ascending=False).drop_duplicates('company_name')

unique_companies.to_csv(r'.\unique_companies_final_dns_pipeline.csv', index=False)